# 07 · V-JEPA 2.1-B continuous CAN v2 — acceleration-collapse fix

This is a **controlled v2 ablation** after notebook 06 diagnosed the first baseline:

- train dynamic fraction was not rare (~35% under the medium proxy rule),
- predicted acceleration standard deviation collapsed to ~7% of GT,
- ACCELERATING / DECELERATING were mapped to CONSTANT almost all the time,
- temporal lag was only a minor effect.

Therefore v2 keeps the **same frozen V-JEPA backbone, 16-frame context, sampled windows, optimizer budget, and base task weights** as v1. Only the continuous-CAN loss formulation changes:

1. magnitude-weighted acceleration regression,
2. multi-threshold signed margin supervision,
3. frame-to-frame speed-delta supervision,
4. a low-weight speed↔acceleration physical consistency term.

`stage3/metrics.py` is **not modified**. The proxy class rules remain diagnostic only and are never described as DACON ground-truth thresholds.

**Important:** this run does **not** warm-start from v1 `best.pt`. It is a separate run from the same pretrained V-JEPA encoder so the comparison is interpretable. If a v2 `latest.pt` already exists, only that v2 run is resumed.


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings(
    "ignore",
    message=r".*torch\.backends\.cuda\.sdp_kernel\(\).*deprecated.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*Importing from timm\.models\.layers is deprecated.*",
    category=FutureWarning,
)

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Initial Drive mount failed; forcing remount:", repr(exc))
    drive.mount("/content/drive", force_remount=True)

try:
    _ = next(Path("/content/drive/MyDrive").iterdir(), None)
except OSError as exc:
    print("Drive mount is stale; forcing remount:", repr(exc))
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current_branch = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current_branch != BRANCH:
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)

    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        print("WARNING: local repo has changes; git pull skipped.")
    else:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )

# Keep Colab's binary stack; install only required extras.
COLAB_EXTRAS = [
    "timm==1.0.15",
    "fvcore==0.1.5.post20221221",
    "iopath==0.1.10",
    "yacs==0.1.8",
    "einops==0.8.1",
    "wandb==0.29.0",
    "easydict==1.13",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.stage3.constants import CAN_TARGETS
from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.utils import (
    dataloader_seed_kwargs,
    finish_wandb,
    init_wandb,
    seed_everything,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
PROCESSED_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
WANDB_KEY_PATH = DRIVE_ROOT / "wandb_key.txt"

LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_OUTPUT_ROOT = Path("/content/stage3_runs")
for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT, LOCAL_PRETRAINED_ROOT, LOCAL_OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CFG_PATH = REPO / "configs" / "stage3" / "vjepa21b_can_accel_v2.yaml"
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
stats = json.loads((MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8"))

SEED = int(cfg["seed"])
seed_everything(SEED, deterministic=False)

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

# Runtime injection keeps the loss tied to the actual split statistics instead
# of duplicating mean/std constants in YAML.
loss_cfg = copy.deepcopy(cfg["loss"])
loss_cfg["normalization"] = {
    name: {
        "mean": float(stats[name]["mean"]),
        "std": float(stats[name]["std"]),
    }
    for name in ("speed_mps", "accel_from_speed_mps2")
}

assert_dacon_metric_contract()

print("Repository     :", REPO)
print("Branch / commit:", BRANCH, "/", GIT_COMMIT)
print("Config         :", CFG_PATH)
print("PROCESSED_ROOT :", PROCESSED_ROOT)
print("torch          :", torch.__version__)
print("cuda           :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu            :", torch.cuda.get_device_name(0))
print("DACON metric contract: PASS (metrics.py unchanged)")
print("loss mode      :", loss_cfg["mode"])
print("accel stats    :", loss_cfg["normalization"]["accel_from_speed_mps2"])


In [ ]:
def _is_usable_file(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= int(min_bytes)
    except OSError:
        return False


def copy_file_to_local(source: Path, destination: Path, *, min_bytes: int = 1, retries: int = 2) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, retries + 2):
        tmp = destination.with_name(destination.name + ".copy.tmp")
        try:
            tmp.unlink(missing_ok=True)
            with source.open("rb") as src, tmp.open("wb") as dst:
                shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
            if tmp.stat().st_size < int(min_bytes):
                raise OSError(f"staged file is too small: {tmp.stat().st_size} bytes")
            os.replace(tmp, destination)
            return True
        except OSError as exc:
            last_error = exc
            tmp.unlink(missing_ok=True)
            print(f"copy attempt {attempt} failed:", repr(exc))
            if attempt <= retries:
                time.sleep(2 * attempt)
    print("Drive -> local copy failed:", repr(last_error))
    return False


VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"
if not (VJEPA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/facebookresearch/vjepa2.git", str(VJEPA_REPO)],
        check=True,
    )
subprocess.run(["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT], check=True)
ACTUAL_VJEPA_COMMIT = subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

CKPT_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
VJEPA_CKPT_DRIVE = PRETRAINED_ROOT / CKPT_NAME
VJEPA_CKPT = LOCAL_PRETRAINED_ROOT / CKPT_NAME
VJEPA_CKPT_URL = "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt"
MIN_VJEPA_BYTES = 1_000_000_000

if not _is_usable_file(VJEPA_CKPT, MIN_VJEPA_BYTES):
    copied = copy_file_to_local(
        VJEPA_CKPT_DRIVE,
        VJEPA_CKPT,
        min_bytes=MIN_VJEPA_BYTES,
    )
    if not copied:
        print("Downloading V-JEPA checkpoint directly to local Colab disk...")
        tmp = VJEPA_CKPT.with_name(VJEPA_CKPT.name + ".download.tmp")
        tmp.unlink(missing_ok=True)
        subprocess.run(
            ["wget", "-q", "--show-progress", "-O", str(tmp), VJEPA_CKPT_URL],
            check=True,
        )
        if tmp.stat().st_size < MIN_VJEPA_BYTES:
            raise RuntimeError(f"Downloaded checkpoint is too small: {tmp.stat().st_size} bytes")
        os.replace(tmp, VJEPA_CKPT)

print("V-JEPA commit            :", ACTUAL_VJEPA_COMMIT)
print("V-JEPA checkpoint (LOCAL):", VJEPA_CKPT)
print("checkpoint size MiB      :", f"{VJEPA_CKPT.stat().st_size / 2**20:.1f}")


In [ ]:
from torch.utils.data import DataLoader
from blackbox_detection.stage3.dataset import Stage3CANDataset

dc = cfg["data"]
tc = cfg["training"]
vc = cfg.get("validation", {})

train_ds = Stage3CANDataset(
    MANIFEST_ROOT / "comma_train.csv",
    PROCESSED_ROOT,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=dc["train_random_flip"],
    max_windows=dc["max_train_windows"],
    seed=SEED,
)

val_ds = Stage3CANDataset(
    MANIFEST_ROOT / "comma_val_id.csv",
    PROCESSED_ROOT,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=False,
    max_windows=dc["max_val_windows"],
    seed=SEED + 1,
)

train_loader = DataLoader(
    train_ds,
    batch_size=tc["batch_size"],
    shuffle=True,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED),
)
val_loader = DataLoader(
    val_ds,
    batch_size=tc["batch_size"],
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED + 1),
)

print("windows             :", len(train_ds), len(val_ds))
print("batch size          :", tc["batch_size"])
print("grad accumulation   :", tc["grad_accum_steps"])
print("effective batch size:", tc["batch_size"] * tc["grad_accum_steps"])
print("num workers         :", dc["num_workers"])


In [ ]:
import importlib

from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.models import VJEPA21DenseCAN
import blackbox_detection.stage3.trainer as trainer_module

trainer_module = importlib.reload(trainer_module)
CANTrainer = trainer_module.CANTrainer
build_parameter_groups = trainer_module.build_parameter_groups
build_scheduler = trainer_module.build_scheduler

mc = cfg["model"]
backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_CKPT,
    num_frames=dc["clip_len"],
    out_layers=tuple(mc["out_layers"]),
    freeze=mc["freeze_backbone"],
)
model = VJEPA21DenseCAN(
    backbone,
    freeze_backbone=mc["freeze_backbone"],
    feature_dim=mc["feature_dim"],
    temporal_hidden=mc["temporal_hidden"],
    temporal_layers=mc["temporal_layers"],
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print("trainable params:", trainable_params / 1e6, "M")
print("total params    :", total_params / 1e6, "M")

optimizer = torch.optim.AdamW(
    build_parameter_groups(
        model,
        learning_rate=tc["learning_rate"],
        weight_decay=tc["weight_decay"],
    )
)

micro_steps_per_epoch = min(len(train_loader), int(tc["max_steps_per_epoch"]))
optimizer_steps_per_epoch = max(
    math.ceil(micro_steps_per_epoch / int(tc["grad_accum_steps"])),
    1,
)
total_optimizer_steps = optimizer_steps_per_epoch * int(tc["epochs"])

scheduler = build_scheduler(
    optimizer,
    total_steps=total_optimizer_steps,
    warmup_ratio=tc["warmup_ratio"],
    min_ratio=tc["min_learning_rate_ratio"],
)

print("micro steps / epoch    :", micro_steps_per_epoch)
print("optimizer steps / epoch:", optimizer_steps_per_epoch)
print("total optimizer steps  :", total_optimizer_steps)

RUN_VARIANT = cfg["experiment"]["name"]
RUN_DIR = OUTPUT_ROOT / RUN_VARIANT
LOCAL_RUN_DIR = LOCAL_OUTPUT_ROOT / RUN_VARIANT
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

# Explicitly preserve v1 as a baseline and never use it as a v2 warm-start.
V1_BEST = OUTPUT_ROOT / "vjepa21b_can_v1" / "best.pt"
print("v1 best exists (comparison only):", V1_BEST.is_file())
print("v1 warm-start used              : False")

# On a fresh runtime, stage ONLY this v2 run's checkpoints before resume.
for filename in ("latest.pt", "best.pt"):
    persistent = RUN_DIR / filename
    local = LOCAL_RUN_DIR / filename
    if not local.is_file() and _is_usable_file(persistent):
        ok = copy_file_to_local(persistent, local, min_bytes=1, retries=2)
        print(f"staged v2 {filename}:", ok, "->", local if ok else None)

lc = cfg.get("logging", {})
WANDB_ENABLED = bool(lc.get("wandb_enabled", True))
wandb_run = None

if WANDB_ENABLED:
    import wandb
    if not WANDB_KEY_PATH.is_file():
        raise FileNotFoundError(f"W&B key file not found: {WANDB_KEY_PATH}")
    wandb_key = WANDB_KEY_PATH.read_text(encoding="utf-8").strip()
    if not wandb_key:
        raise ValueError(f"W&B key file is empty: {WANDB_KEY_PATH}")
    wandb.login(key=wandb_key, relogin=False)
    del wandb_key
    finish_wandb()

    run_id_path = RUN_DIR / "wandb_run_id.txt"
    stored_run_id = run_id_path.read_text(encoding="utf-8").strip() if run_id_path.is_file() else None
    stored_run_id = stored_run_id or None

    run_name = f"{RUN_VARIANT}__seed{SEED}__{GIT_COMMIT}"
    wandb_run = init_wandb(
        project=str(lc.get("wandb_project", "blackbox-stage3")),
        entity=os.getenv("WANDB_ENTITY") or None,
        name=run_name,
        group=str(lc.get("wandb_group", "vjepa21b_can_accel_v2")),
        tags=["stage3", "vjepa2", "comma2k19", "continuous-can", "accel-v2", "frozen-backbone"],
        run_id=stored_run_id,
        resume="allow",
        config={
            "git_commit": GIT_COMMIT,
            "vjepa_commit": ACTUAL_VJEPA_COMMIT,
            "run_variant": RUN_VARIANT,
            "seed": SEED,
            "num_train_windows": len(train_ds),
            "num_val_windows": len(val_ds),
            "trainable_params": trainable_params,
            "total_params": total_params,
            "effective_batch_size": tc["batch_size"] * tc["grad_accum_steps"],
            "config": cfg,
            "runtime_loss_config": loss_cfg,
        },
        directory=LOCAL_RUN_DIR / "wandb",
        mode=os.getenv("WANDB_MODE") or None,
    )
    if stored_run_id is None:
        run_id_path.write_text(wandb_run.id, encoding="utf-8")
    print("W&B run:", wandb_run.name)
    print("W&B id :", wandb_run.id)
    print("W&B url:", wandb_run.url)

trainer_config = {
    "git_commit": GIT_COMMIT,
    "vjepa_commit": ACTUAL_VJEPA_COMMIT,
    "run_variant": RUN_VARIANT,
    "seed": SEED,
    "data": dc,
    "model": mc,
    "training": tc,
    "loss": loss_cfg,
    "validation": vc,
    "logging": lc,
    "warm_start_from_v1": False,
}

trainer = CANTrainer(
    model,
    optimizer,
    scheduler=scheduler,
    grad_accum_steps=tc["grad_accum_steps"],
    grad_clip_norm=tc["grad_clip_norm"],
    amp_dtype=tc["amp_dtype"],
    loss_weights=loss_cfg,
    stats=stats,
    proxy_rules=vc.get("proxy_rules", {}),
    output_dir=LOCAL_RUN_DIR,
    sync_dir=RUN_DIR,
    wandb_enabled=WANDB_ENABLED,
    log_interval=tc.get("log_interval", 20),
    config=trainer_config,
)

print("local run dir     :", LOCAL_RUN_DIR)
print("persistent run dir:", RUN_DIR)


In [ ]:
# Quick one-batch forward/loss smoke before spending an epoch.
from blackbox_detection.stage3.losses import can_multitask_loss

smoke = next(iter(train_loader))
video = smoke["video"][:1].to(trainer.device)
target = smoke["target"][:1].to(trainer.device)
valid = smoke["valid"][:1].to(trainer.device)

model.train()
with torch.autocast(
    device_type=trainer.device.type,
    dtype=trainer.amp_dtype,
    enabled=trainer.device.type == "cuda",
):
    smoke_out = model(video)
    smoke_loss, smoke_parts = can_multitask_loss(
        smoke_out,
        target,
        valid,
        loss_cfg,
    )

if not torch.isfinite(smoke_loss):
    raise RuntimeError(f"Non-finite v2 smoke loss: {smoke_loss}")

print("v2 smoke total:", float(smoke_loss.detach().cpu()))
print(json.dumps(smoke_parts, indent=2))

# Free smoke tensors before the real loop.
del video, target, valid, smoke_out, smoke_loss
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
resume_path = LOCAL_RUN_DIR / "latest.pt"
print("resume:", resume_path if resume_path.is_file() else None)

history = trainer.fit(
    train_loader,
    val_loader,
    epochs=tc["epochs"],
    max_train_steps=tc["max_steps_per_epoch"],
    max_val_steps=tc.get("max_val_steps", 500),
    resume_from=resume_path if resume_path.is_file() else None,
    early_stopping_patience=tc.get("early_stopping_patience", 0),
    backfill_validation_on_resume=bool(vc.get("backfill_on_resume", True)),
)

history_df = pd.DataFrame([
    {
        "epoch": x["epoch"],
        "minutes": x["minutes"],
        "learning_rate": x.get("learning_rate"),
        "global_step": x.get("global_step"),
        "max_gpu_memory_gib": x.get("max_gpu_memory_gib"),
        **{f"train/{k}": v for k, v in x["train"].items()},
        **{f"val/{k}": v for k, v in x["val"].items()},
    }
    for x in history
])

display(history_df)


In [ ]:
if len(history_df):
    best_idx = history_df["val/total"].astype(float).idxmin()
    best_row = history_df.loc[best_idx]

    summary = {
        "run_variant": RUN_VARIANT,
        "git_commit": GIT_COMMIT,
        "vjepa_commit": ACTUAL_VJEPA_COMMIT,
        "best_epoch": int(best_row["epoch"]),
        "best_val_total": float(best_row["val/total"]),
        "checkpoint_selection": "minimum v2 val/total; proxy metrics remain diagnostic",
        "proxy_metric_status": "diagnostic only; not official DACON labels/thresholds",
        "warm_start_from_v1": False,
        "num_train_windows": int(len(train_ds)),
        "num_val_windows": int(len(val_ds)),
        "trainable_params": int(trainable_params),
    }

    for column, value in best_row.items():
        if isinstance(column, str) and column.startswith("val/"):
            try:
                summary[f"best_{column}"] = float(value)
            except (TypeError, ValueError):
                pass

    proxy_col = "val/proxy/robust_mean_stage3_score"
    if proxy_col in history_df.columns:
        s = pd.to_numeric(history_df[proxy_col], errors="coerce")
        if s.notna().any():
            i = s.idxmax()
            summary["diagnostic_best_proxy_epoch"] = int(history_df.loc[i, "epoch"])
            summary["diagnostic_best_proxy_mean_stage3_score"] = float(s.loc[i])

    # Compare directly to the notebook-06 v1 failure signature when available.
    v1_diag_path = OUTPUT_ROOT / "vjepa21b_can_v1" / "diagnostics_accel_v1" / "diagnostic_summary.json"
    if v1_diag_path.is_file():
        try:
            v1_diag = json.loads(v1_diag_path.read_text(encoding="utf-8"))
            summary["v1_diag_accel_pred_to_gt_std_ratio"] = float(v1_diag["accel_pred_to_gt_std_ratio"])
            summary["v1_diag_accel_prediction_slope"] = float(v1_diag["accel_prediction_slope"])
            summary["v1_diag_dynamic_to_constant_confusion_mean"] = float(v1_diag["dynamic_to_constant_confusion_mean"])
        except Exception as exc:
            print("v1 diagnostic comparison warning:", repr(exc))

    local_summary = LOCAL_RUN_DIR / "summary.json"
    local_summary.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    trainer._sync_file(local_summary)

    print(json.dumps(summary, indent=2))

    # Compact success readout. These are diagnostics, not DACON score claims.
    diag_cols = [
        "val/diag/accel/pred_to_gt_std_ratio",
        "val/diag/accel/pred_vs_gt_slope",
        "val/diag/accel/correlation",
        "val/proxy/medium/dynamic_to_constant_rate",
        "val/proxy/robust_mean_accel_macro_f1",
        "val/proxy/robust_mean_stage3_score",
    ]
    present = [c for c in diag_cols if c in history_df.columns]
    if present:
        print("\nV2 acceleration diagnostics by epoch:")
        display(history_df[["epoch", *present]])

    if WANDB_ENABLED and wandb_run is not None:
        for key, value in summary.items():
            if isinstance(value, (str, int, float, bool)) or value is None:
                wandb_run.summary[key] = value
        try:
            for filename in ("summary.json", "history.csv", "train_config.json", "train.log"):
                path = LOCAL_RUN_DIR / filename
                if path.is_file():
                    wandb_run.save(str(path), base_path=str(LOCAL_RUN_DIR))
        except Exception as exc:
            print("W&B file upload warning:", repr(exc))

if WANDB_ENABLED:
    finish_wandb()
    print("W&B run finished.")

print("local latest     :", LOCAL_RUN_DIR / "latest.pt")
print("persistent latest:", RUN_DIR / "latest.pt")
print("persistent best  :", RUN_DIR / "best.pt")


## How to judge v2

The main purpose of this run is to test whether the diagnosed longitudinal-collapse failure is reversible **without changing the backbone or data budget**.

The most important diagnostics are:

- `val/diag/accel/pred_to_gt_std_ratio` — v1 was about **0.071**; this should rise materially.
- `val/diag/accel/pred_vs_gt_slope` — v1 was about **0.017**; this should rise materially.
- `val/proxy/medium/dynamic_to_constant_rate` — v1 was about **0.994**; this should fall strongly.
- `val/diag/accel/correlation` — v1 was about **0.240**.
- `val/proxy/robust_mean_accel_macro_f1` — diagnostic only, but useful to check whether the continuous fix improves class-consistent behavior across several non-official thresholds.

Do not interpret the proxy score as the DACON leaderboard score. `metrics.py` remains the only source of truth for official Stage-3 scoring semantics.

If v2 still produces a very small accel standard-deviation ratio despite these losses, the next experiment should unfreeze the last V-JEPA blocks (or use adapters/LoRA) rather than merely adding more epochs.
